In [ ]:
# capstone project -- function 6 (5D), week N

import numpy as np
import matplotlib.pyplot as plt

from scipy.spatial import Delaunay, ConvexHull
from scipy.stats import spearmanr

from bayes_tools import (
    fitting,
    normalize, initial_bounds, validate_bounds_consistency,
    generate_next_point, ucb_acquisition,
    append_observations, compute_iteration_diagnostics,
    fit_gp, get_length_scales, loo_predictions,
    compare_kappa_proposals, print_kappa_comparison,
    backtest_acquisitions, print_backtest_summary,
)
from viz_tools import (
    plot_nd_slices, plot_loo_calibration, plot_kappa_sensitivity,
    plot_acquisition_backtest,
    plot_convergence, plot_acquisition_decay, plot_uncertainty_shrinkage,
    plot_step_distance,
)

# Function 6

5D, 23 observations (20 initial + weeks 2–4), all used for modelling. **Acquisition is `ucb` with `kappa = 0.25`.** Search domain is the unit cube `[0,1]^5`.

Proposal this week: **`[0.518201, 0.321942, 0.527558, 0.786655, 0.000000]`** (GP mean −0.3832, std 0.0343, predicted improvement **+0.0029** on an incumbent of −0.386121).

## Week 4 worked, and the floor contact was right

`[0.497078, 0.264731, 0.582415, 0.764018, 0.0]` returned **−0.386121** — a new best. Progress so far: initial −0.7143 → wk2 −0.5120 → wk3 −0.5120 (no gain; that point was out of domain) → wk4 **−0.3861**.

Its `x4 = 0.0` sat on the lower bound, which the previous version of this notebook treated as merely "legitimate". It is better than that: **x4 is monotone decreasing, so 0 is its optimal end.** Spearman(x4, y) = −0.654, the low-x4 half of the data has mean `y` −1.148 against −1.616 for the high half, and the five highest-x4 observations run −1.67 to −2.57. Every acquisition tried — `exploit`, `ucb` at 0, 0.25 and 0.5, EI and PI at small `xi` — independently puts x4 at 0. So the floor contact is the model agreeing with the data, not an artefact.

## No axis is held: all five are live

Unlike functions 3, 5, 7 and 8, this notebook holds nothing fixed. The axis cell selects flat axes by **GP mean-variation below 5% of the maximum, or a length-scale pinned at the `length_scale_bounds` ceiling** — and neither fires:

| axis | mean-variation | length-scale | Spearman | reading |
|---|---|---|---|---|
| x0 | 1.132 | 0.595 | −0.030 | non-monotone |
| x1 | 0.666 | 1.024 | −0.290 | non-monotone |
| x2 | 0.294 | 1.253 | +0.183 | weak |
| x3 | 0.825 | 1.062 | +0.602 | monotone |
| x4 | 0.958 | 1.225 | −0.654 | monotone |

Worth being clear why the length-scale column alone would mislead here. Four of these exceed the domain width of 1.0, which looks like "flat" — but a long length-scale means *no short-range structure*, not *no effect*. x4's length-scale is 1.225 and it moves the posterior mean by 0.958, nearly as much as x0. The mean-variation column is what separates the two cases; the pinned check (length-scale at 10.0, as on functions 2, 3 and 5) is a genuinely different and stronger signal, and nothing here is close to it.

## Why `kappa = 0.25`

The previous rationale — *"the largest kappa whose proposal touches no UPPER bound"* — rested on upper-bound contact meaning `pad_fraction` was choosing the point. That was true of padded bounds and is **false now**: bounds are the true domain, so an upper-bound contact would mean the constrained optimum is on the boundary. The criterion had to be replaced.

The replacement is checkable: **the largest `kappa` whose predicted mean still beats the incumbent.**

| kappa | proposal | pred. mean | gain vs incumbent | step |
|---|---|---|---|---|
| 0 (= `exploit`) | `[0.5129, 0.3003, 0.5507, 0.7758, 0.0]` | −0.38185 | **+0.00427** | 0.052 |
| **0.25** | `[0.5182, 0.3219, 0.5276, 0.7867, 0.0]` | −0.38325 | **+0.00287** | 0.085 |
| 0.5 (previous) | `[0.5115, 0.3283, 0.4641, 0.8025, 0.0]` | −0.38952 | **−0.00340** | 0.141 |

The committed `0.5` was predicting a *loss*. `0.25` buys a 65% larger step than pure `exploit` for a third less predicted gain, and keeps a variance term — function 4 showed what happens when `exploit` runs with no exploration term at all and stalls to re-proposing the incumbent. A mode switch sits between `kappa` 1.0 and 1.5 (x0 jumps 0.56 → 0.36), so anything above 1 is off the table regardless.

The backtest agrees only weakly, as expected: `exploit` leads (0.4025) with `ucb_k0.5` (+0.043 ± 0.048) and `ucb_k1` (+0.094 ± 0.053) within noise. It is an exploitation-biased metric, so its ranking is close to tautological — the predicted-gain table above is the reason.

## EI and PI: `xi` is live here, and they still lose

Unlike function 8, where every `xi` across a 25× range returned the identical corner, `xi` genuinely moves the proposal on this function — EI travels 0.007 → 0.036 → 1.156 → 1.693 as `xi` rises from 0.1% to 100% of the `y` spread. But both families head for bad regions (predicted −0.62 to −1.71), so they are variance-seekers here rather than degenerate. The backtest puts them 2.5 s.e. (PI) and 4.4 s.e. (EI) behind. Only PI at `xi` ≈ 0.1% of spread comes close to `exploit`, and then it is just `exploit` with extra steps.

## What to watch

- **Local search is close to finished.** Every axis subset and every acquisition lands within ±0.01 of the incumbent, and the largest gap on the most informative axis (x0) is 0.137 against a length-scale of 0.595 — a ratio of **0.23**, well below the 1.0 that would indicate unresolved structure.
- **But almost nothing is explored.** The convex hull is **6.4%** of the cube, 22 of 23 points are hull vertices (so the hull test is near-vacuous at D=5), and posterior std at the incumbent is 0.0007 against a domain median of 0.210 — **100%** of the domain has more than twice the incumbent's uncertainty. Genuine exploration goes to corners: `max_variance` proposes `[0,0,0,0,0]` at a predicted −1.647.
- So if this week's step fails to beat −0.386121, that is the signal to spend a deliberate exploration budget rather than to keep polishing. Do not read a small miss as "lower kappa further".
- **The out-of-domain week-3 point stays in the fit.** It shortens length-scales (0.44–0.67×) rather than inflating them, and without it four of five would exceed the domain width. Contrast function 5, where out-of-domain `y` was 53× the in-domain best and had to be excluded.
- LOO: R² **+0.590**, **83.4%** of point pairs ordered correctly, 19/23 inside their own 95% interval.


In [16]:
X_initial = np.load("initial_data/function_6/initial_inputs.npy")
y_initial = np.load("initial_data/function_6/initial_outputs.npy")
n_initial = len(y_initial)
D = X_initial.shape[1]

assert D == 5, f"Expected a 5D problem, got {D}D input -- check the loaded file."

# Once, at the very start of the capstone:
new_X = np.empty((0, D))
new_y = np.empty((0,))

## Update this once per week

Include the new X and y values from the previous week, oldest first -- row order must be true chronological order, or `compute_iteration_diagnostics` at the bottom is meaningless.

One observation is recorded, and it is the new best. Clean provenance: the old sweep proposed exactly these coordinates, all positive, none on a bound, and every one inside the range already sampled on its axis.

In [ ]:
# Append last week's result BEFORE proposing this week's point, e.g.:
#
# new_X, new_y = append_observations(new_X, new_y, x_next, the_result_you_got)

new_X = np.array([
    [0.519691, 0.384667, 0.382026, 0.764661, 0.218558],     # week 2
    [0.371752, 0.146637, 0.75637, 1.545378, 0.0], 
    [0.497078, 0.264731, 0.582415, 0.764018, 0.      ],          # week 3 -- x4 at the floor
])

new_y = np.array([
    -0.5119821356901731,
    -1.2147961781405971,
    -0.3861205609344592,
])

print("observations collected so far:", len(new_y))
print("latest y: %.6f | best before it: %.6f (change %+.6f)"
      % (new_y[-1], max(y_initial.max(), new_y[0]),
         new_y[-1] - max(y_initial.max(), new_y[0])))
print("-> %s" % ("the WORST observation on record"
                 if new_y[-1] < min(y_initial.min(), new_y[0])
                 else "within the existing range"))


observations collected so far: 2
latest y: -1.214796 | best before it: -0.511982 (change -0.702814)
-> within the existing range


## Build the full dataset (initial + everything collected so far)

In [ ]:
X, y = append_observations(X_initial, y_initial, new_X, new_y)

print("X_initial shape:", X_initial.shape, "| combined X shape:", X.shape)
print("y range: %.6f to %.6f (spread %.6f, std %.6f)"
      % (y.min(), y.max(), y.max() - y.min(), y.std()))

# X is known to never be negative -- lower_limit=0.0 is mandatory. Note this
# makes the LOWER bound a real domain constraint, while the upper bounds are
# an artifact of pad_fraction. The checks below treat them differently.
bounds = initial_bounds(X_initial, pad_fraction=1.0, lower_limit=0.0,
                        upper_limit=1.0)
# Bounds ARE the domain now, so some observations legitimately sit outside them:
# earlier weeks proposed points above 1 before the domain was known. Validate the
# in-domain subset -- catching bounds too narrow for the data is what
# validate_bounds_consistency is for -- and report the rest instead of dying on
# them.
_in_dom = np.all((X >= bounds[:, 0]) & (X <= bounds[:, 1]), axis=1)
validate_bounds_consistency(X[_in_dom], bounds)

if not _in_dom.all():
    print(f"\n{int((~_in_dom).sum())} observation(s) lie OUTSIDE the [0,1] domain,"
          " proposed before it was known:")
    for i in np.flatnonzero(~_in_dom):
        over = [f"x{d}={X[i, d]:.4f}" for d in range(D) if X[i, d] > 1.0]
        print(f"  {', '.join(over)}   y={y[i]:+.5g}")
    print("  They stay IN the fit. Their effect on the kernel was measured and is")
    print("  negligible-or-beneficial here (they SHORTEN length-scales, i.e. the GP")
    print("  sees more structure), and their y values are ordinary. Contrast")
    print("  function 5, where out-of-domain y was 53x the in-domain best and had")
    print("  to be excluded. The SEARCH is clamped to [0,1] either way, so nothing")
    print("  can be proposed out there again.")
print("\nBounds:\n", np.round(bounds, 4))

print("\ncollected points vs the INITIAL batch's range:")
for r in range(len(new_X)):
    n_out = 0
    for d in range(D):
        outside = (new_X[r, d] > X_initial[:, d].max()
                   or new_X[r, d] < X_initial[:, d].min())
        n_out += outside
        print(f"  week {r + 2}  x{d}={new_X[r, d]:.6f}  initial range"
              f" [{X_initial[:, d].min():.4f}, {X_initial[:, d].max():.4f}]"
              f"  outside={outside}")
    print(f"  -> week {r + 2}: {n_out} of {D} coordinates outside,"
          f" y={new_y[r]:.6f}")

# The convex hull is a weak notion at D=5 with this many points -- quantify how
# weak, so the check below is read with the right expectations.
try:
    hull = Delaunay(X)
    ch = ConvexHull(X)
    box_volume = float(np.prod(X.max(axis=0) - X.min(axis=0)))
    print(f"\nobserved box volume {box_volume:.4g} | convex hull {ch.volume:.4g}"
          f" ({ch.volume / box_volume:.1%} of it)")
    print(f"{len(ch.vertices)} of {len(X)} observations are hull vertices"
          " -- at D=5 nearly all of them are, so the hull test is weak here")
except Exception as exc:
    hull = None
    print("\nconvex hull unavailable:", exc)

xi_frac = 0.01 / (y.max() - y.min())
print(f"\nxi=0.01 is {xi_frac:.4%} of the y spread"
      f" -> {'scaling NOT needed' if 0.001 <= xi_frac <= 0.1 else 'CONSIDER y-scaling'}")
print("(kappa is dimensionless, so UCB is unaffected either way)")

## Is the model stable and calibrated?

Two checks that were load-bearing on functions 4 and 5, run here as confirmation.

First, refit on the initial batch alone and predict the collected point. That fit never saw it, so this is a genuine out-of-sample test; the miss in units of its own predicted sigma says how much to trust the model's uncertainty. On function 5 this came back at 17 sigma and changed the whole approach.

Second, compare kernels before and after. One observation reshaping the model is a reason to distrust any variance-driven acquisition. Function 4's length-scales tripled from a single point; expect this one to barely move.

In [ ]:
with fitting("before/after fits for the new observation"):
    gp_before = fit_gp(X_initial, y_initial, bounds, n_restarts_optimizer=25, random_state=0)

with fitting("before/after fits for the new observation"):
    gp_check = fit_gp(X, y, bounds, n_restarts_optimizer=25, random_state=0)


# Fit is the initial batch only, so every collected row is out of sample. The
# model that actually proposed week 3 also had week 2, so this is a lower
# bound on what was known at proposal time for later rows.
mu_b, sd_b = gp_before.predict(normalize(new_X, bounds), return_std=True)
print("out-of-sample test on the collected points (fit = initial batch only):")
for r in range(len(new_X)):
    miss = new_y[r] - mu_b[r]
    print(f"  week {r + 2}: predicted {mu_b[r]:+.6f} +/- {sd_b[r]:.6f}"
          f"   actual {new_y[r]:+.6f}")
    print(f"    miss {miss:+.6f} = {miss / sd_b[r]:+.1f} sigma"
          f"  -> {'well calibrated' if abs(miss / sd_b[r]) < 3 else 'POORLY calibrated'}")

ls_before, ls_after = get_length_scales(gp_before), get_length_scales(gp_check)
print("\nkernel before:", gp_before.kernel_)
print("kernel after :", gp_check.kernel_)
print("\nlength-scales before:", np.round(ls_before, 3))
print("length-scales after :", np.round(ls_after, 3))
print("ratio (after/before):", np.round(ls_after / ls_before, 2))

PINNED = 10.0  # fit_gp's default length_scale_bounds upper limit
print("\npinned (GP treats as irrelevant) before:",
      [d for d, v in enumerate(ls_before) if v >= 0.999 * PINNED] or "none")
print("pinned after                          :",
      [d for d, v in enumerate(ls_after) if v >= 0.999 * PINNED] or "none")

if np.max(np.abs(ls_after / ls_before - 1)) > 0.5:
    print("\n*** A length-scale moved by more than 50% from one observation.")
    print("    Treat variance-driven acquisitions with suspicion this week. ***")


## Which axes matter, and in what way

Two complementary measures, because they answer different questions.

**GP mean-variation** (sweep one axis, others held at the incumbent) asks "how much does the model think this axis moves the response, in any pattern". **Spearman rank correlation** asks "is there a usable monotone direction". An axis can score high on the first and near-zero on the second: that is a non-monotone effect, and it is exactly what x0 shows here (variation 2.02, the shortest length-scale, but rank correlation -0.018). Not a contradiction -- Spearman is blind to non-monotone structure by construction.

The cell also applies the flat-axis rule used on functions 3, 5, 7 and 8: an axis counts as flat if its mean-variation is under 5% of the largest, **or** its length-scale is pinned near the 10.0 ceiling. Flat axes get held at the incumbent instead of searched, because a flat acquisition surface makes their coordinates optimiser artifacts. On this function nothing qualifies, so all five axes are searched -- which is why there is no restricted-search cell here, unlike functions 3, 5, 7 and 8.

In [ ]:
incumbent = X[np.argmax(y)]
print("incumbent (best observed):", np.round(incumbent, 6), " y = %.6f" % y.max())

print("\n%3s | %17s | %15s | %8s | %9s | %s"
      % ("ax", "GP mean-variation", "same, observed", "len-scale", "spearman", "reading"))
spans = np.zeros(D)
for d in range(D):
    row = []
    for lo, hi in [(bounds[d, 0], bounds[d, 1]), (X[:, d].min(), X[:, d].max())]:
        grid = np.tile(incumbent, (300, 1))
        grid[:, d] = np.linspace(lo, hi, 300)
        m, _ = gp_check.predict(normalize(grid, bounds), return_std=True)
        row.append(m.max() - m.min())
    spans[d] = row[0]
    sr = spearmanr(X[:, d], y)[0]
    reading = "monotone direction" if abs(sr) > 0.4 else (
        "non-monotone" if row[0] > 0.5 * spans.max() else "weak")
    print("%3d | %17.4g | %15.4g | %8.3f | %+9.3f | %s"
          % (d, row[0], row[1], ls_after[d], sr, reading))

FLAT_FRAC = 0.05
flat = [d for d in range(D)
        if spans[d] < FLAT_FRAC * spans.max() or ls_after[d] >= 0.9 * PINNED]
live = [d for d in range(D) if d not in flat]
print(f"\nflat axes (mean-variation < {FLAT_FRAC:.0%} of max, or length-scale >= {0.9 * PINNED}):",
      flat or "none")
print("axes searched:", live)
print("dominance: x%d moves the mean %.1fx more than the next-largest (x%d)"
      % (int(np.argmax(spans)), spans[np.argsort(spans)[-1]] / spans[np.argsort(spans)[-2]],
         int(np.argsort(spans)[-2])))
# A monotone axis whose best end is a domain bound will put the proposal ON that
# bound. That is the model agreeing with the data, not the failure mode the
# proposal cell warns about -- flag it here so the two are not confused.
for d in range(D):
    sr = spearmanr(X[:, d], y)[0]
    if abs(sr) > 0.4:
        end = bounds[d, 0] if sr < 0 else bounds[d, 1]
        which = "LOWER" if sr < 0 else "UPPER"
        print(f"  x{d} is monotone (spearman {sr:+.3f}): its best end is the {which}"
              f" bound ({end:.1f}), so a proposal sitting there is expected")

if not flat:
    print("-> nothing to hold fixed: all axes are live, so the search is unrestricted")
    print("   (contrast functions 3, 5, 7 and 8, which each hold one or more axes)")

## Backtest acquisition functions (using only data already collected)

Repeatedly splits the data into a "seed" set (fits the GP) and a held-out "candidate" set (true y known, hidden from the fit), then sees which config would have picked the best candidate most often. No new evaluations spent. `xi` is sized as 1% of the `y` spread rather than the default 0.01.

**Biased toward exploitation by construction** -- it scores recognition of points whose `y` is already known, which rewards ranking by posterior mean and gives no credit for reducing uncertainty. So it favours `exploit` and low `kappa` on every function, and it ranked function 3's chosen config last. Use it as one weak input.

Here it happens to agree with the upper-bound argument in the header: expect low `kappa` at the top, `max_variance` last, and the top two within about 0.03 of each other with identical medians. Agreement, not proof.

In [ ]:
xi_raw = 0.01 * (y.max() - y.min())
backtest_configs = [
    {"name": "ucb_k0.25", "acquisition": "ucb", "kappa": 0.25},
    {"name": "ucb_k0.5", "acquisition": "ucb", "kappa": 0.5},
    {"name": "ucb_k1",   "acquisition": "ucb", "kappa": 1.0},
    {"name": "ucb_k2",   "acquisition": "ucb", "kappa": 2.0},
    {"name": "ucb_k5",   "acquisition": "ucb", "kappa": 5.0},
    {"name": "ei",       "acquisition": "ei",  "xi": xi_raw},
    {"name": "pi",       "acquisition": "pi",  "xi": xi_raw},
    {"name": "exploit",  "acquisition": "exploit"},
    {"name": "max_var",  "acquisition": "max_variance"},
]
print(f"xi for the PI/EI rows: {xi_raw:.6f} (1% of the y spread)")

# Is xi a live dial here, or degenerate as on function 8? Checked rather than
# assumed: on function 8 every xi across a 25x range returned the identical
# corner. Here it genuinely moves the proposal -- but toward bad regions.
print("\nis xi a live dial? (proposal movement as xi grows)")
_prev = {}
for _acq in ("ei", "pi"):
    for _frac in (0.001, 0.05, 1.0):
        _xi = _frac * (y.max() - y.min())
        with fitting(f"{_acq} at xi={_frac:.1%} of spread"):
            _p, _ = generate_next_point(X, y, bounds, acquisition=_acq, xi=_xi,
                                        maximize=True, n_restarts=40, random_state=0)
        _m, _ = gp_check.predict(normalize(_p.reshape(1, -1), bounds), return_std=True)
        _mv = "" if _acq not in _prev else f"  moved {np.linalg.norm(_p - _prev[_acq]):.4f}"
        print(f"  {_acq} xi={_frac:>6.1%} of spread: pred mean {_m[0]:+.4f}{_mv}")
        _prev[_acq] = _p
print("  -> xi IS live (unlike function 8), but both families head for predicted")
print("     means far below the incumbent, so they are variance-seeking here.\n")

with fitting("acquisition backtest, 50 splits"):
    backtest_results = backtest_acquisitions(
        X, y, bounds, backtest_configs,
        n_repeats=50, seed_frac=0.5, maximize=True,
        gp_kwargs={"n_restarts_optimizer": 15}, random_state=0,
    )

print_backtest_summary(backtest_results)

# ---------------------------------------------------------------------------
# Which acquisition appears best on the evidence so far?
#
# Every config is scored on the SAME splits, so they are compared PAIRED: the
# per-split difference in regret is far less noisy than the two means
# separately, and its standard error says whether a gap is real. "Within noise"
# means |mean difference| <= 2 standard errors. Plain arithmetic, no test.
#
# Read it knowing the metric's bias (CLAUDE.md): it scores RECOGNITION of points
# whose y is already known, which is an exploitation task. It structurally
# favours `exploit` and low `kappa` and penalises `max_variance` on every
# function, so a low-kappa config at the top is close to tautological.
# ---------------------------------------------------------------------------
CHOSEN = "ucb_k0.25"        # must name the committed KAPPA; asserted where KAPPA is set

assert CHOSEN in backtest_results, (
    f"{CHOSEN!r} is not among the backtest configs, so the committed setting is "
    f"never scored. Add a row for it. Have: {sorted(backtest_results)}"
)

names = list(backtest_results)
mean_r = {k: float(backtest_results[k]["regret"].mean()) for k in names}
med_r = {k: float(np.median(backtest_results[k]["regret"])) for k in names}
ranked = sorted(names, key=lambda k: mean_r[k])
leader = ranked[0]
n_splits = len(backtest_results[leader]["regret"])


def paired_gap(a, b):
    """Mean per-split regret difference (b - a), and its standard error."""
    d = backtest_results[b]["regret"] - backtest_results[a]["regret"]
    return float(d.mean()), float(d.std(ddof=1) / np.sqrt(len(d)))


tied = [k for k in ranked[1:]
        if abs(paired_gap(leader, k)[0]) <= 2 * paired_gap(leader, k)[1]]
worse = [k for k in ranked[1:] if k not in tied]

print("\n=== which acquisition appears best in our tests so far? ===")
print(f"leader by mean regret   : {leader:>10}  ({mean_r[leader]:.4f})")
best_med = min(names, key=lambda k: med_r[k])
print(f"leader by median regret : {best_med:>10}  ({med_r[best_med]:.4f})"
      + ("   (agrees)" if best_med == leader else "   (DISAGREES with the mean)"))

print(f"\npaired against {leader}, over the same {n_splits} splits:")
for k in ranked[1:]:
    gap, se = paired_gap(leader, k)
    print(f"  {k:>10}: {gap:+.4f} +/- {se:.4f} ({gap / se:>5.1f} s.e.)"
          f"   {'within noise' if k in tied else 'clearly worse'}")

print(f"\nindistinguishable from the leader : {', '.join([leader] + tied)}")
print(f"clearly worse                     : {', '.join(worse) or 'none'}")

rank = ranked.index(CHOSEN) + 1
gap, se = paired_gap(leader, CHOSEN)
print(f"\nthis notebook proposes with {CHOSEN}: ranked {rank} of {len(ranked)}", end="")
if CHOSEN == leader:
    print(" -- it leads.")
else:
    print(f", {gap:+.4f} +/- {se:.4f} behind {leader}"
          f" ({'within noise' if CHOSEN in tied else 'a REAL gap'}).")
print("Do NOT switch on this table alone -- see the bias note above, and check")
print("where each candidate would actually propose before acting on it.")

plot_acquisition_backtest(backtest_results)
plt.show()


## Compare kappa values -- the deciding cell

Fits ONE shared GP and shows where each candidate `kappa` would propose to search *right now*, so differences between rows are purely down to `kappa` rather than GP-refit noise.

The criterion used here is **upper-bound contact**. `kappa=0.5` is the largest value whose proposal touches no upper bound; at `kappa >= 1` the proposal starts pinning coordinates to `pad_fraction`'s arbitrary ceiling, at which point the padding rather than the model is choosing them.

Lower-bound contact is reported separately and is **not** disqualifying: every axis has a lower bound of `0.0`, which is a real domain constraint, so a coordinate sitting there is a genuine "as low as allowed" prediction. x4 does exactly that, and independently has the strongest negative rank correlation with `y` (-0.609).

In [ ]:
with fitting("kappa sweep (one shared GP)"):
    kappa_rows = compare_kappa_proposals(X, y, bounds, kappa_values=[0.0, 0.25, 0.5, 1.0, 2.0, 5.0],
                                          maximize=True, n_restarts=40, random_state=0)

print_kappa_comparison(kappa_rows)


def bound_report(p):
    """Report which coordinates sit on a bound.

    NOTE the reading changed once bounds became the true domain [0,1]. Under the
    old padded bounds, upper-bound contact meant pad_fraction was choosing the
    point rather than the model. Now BOTH edges are real domain limits, so
    contact means the constrained optimum is on the boundary -- which on a
    monotone axis is exactly right (see the axis cell). Neither edge is
    automatically a defect any more; judge it per axis.
    """
    upper = [d for d in range(D) if np.isclose(p[d], bounds[d, 1])]
    lower = [d for d in range(D) if np.isclose(p[d], bounds[d, 0])]
    return upper, lower


print(f"\n{'kappa':>6} | {'UPPER bound (artifact)':>22} | {'lower bound (real floor)':>24} | in hull")
for row in kappa_rows:
    up, lo = bound_report(row["x_next"])
    ih = bool(hull.find_simplex(row["x_next"]) >= 0) if hull is not None else None
    print(f"{row['kappa']:6g} | {str(up) if up else 'none':>22}"
          f" | {str(lo) if lo else 'none':>24} | {ih}")

plot_kappa_sensitivity(X, y, bounds, gp_check, kappa_rows, ucb_acquisition, maximize=True)
plt.show()

# Committed choice -- reused by the proposal, the slice plot, and the diagnostics
# replay below, so they can't silently drift apart.
# Chosen as the largest kappa whose predicted mean still BEATS the incumbent.
# The previous criterion -- "the largest kappa touching no upper bound" -- was
# built on upper contact meaning a pad_fraction artifact, which is false now
# that bounds are the domain. At kappa=0.5 the predicted gain is negative.
KAPPA = 0.25
print(f"\nusing kappa = {KAPPA} (dimensionless -- no y-scaling needed)")
# All four fits first, inside one fitting() block, then print -- otherwise each
# fit's ConvergenceWarnings land between the table rows.
_gain_rows = []
with fitting("predicted-gain check across kappa"):
    for _k in (0.0, 0.25, 0.5, 1.0):
        _p, _ = generate_next_point(X, y, bounds, acquisition="ucb", kappa=_k,
                                    maximize=True, n_restarts=40, random_state=0)
        _m, _ = gp_check.predict(normalize(_p.reshape(1, -1), bounds), return_std=True)
        _gain_rows.append((_k, _m[0], np.linalg.norm(_p - incumbent)))

print("chosen as the largest kappa whose predicted mean still beats the incumbent:")
for _k, _mean, _step in _gain_rows:
    _mark = "  <- committed" if _k == KAPPA else ""
    print(f"  kappa={_k:<5} pred mean {_mean:+.6f}  gain {_mean - y.max():+.6f}"
          f"  step {_step:.4f}{_mark}")
print("  (a mode switch sits between kappa 1.0 and 1.5, so >1 is off the table)")

# The verdict cell above reports how the committed setting ranks, which only
# means something if CHOSEN names this KAPPA. Config names follow
# f"ucb_k{kappa:g}", so this is checkable rather than a comment to remember.
_expected = f"ucb_k{KAPPA:g}"
assert CHOSEN == _expected, (
    f"CHOSEN={CHOSEN!r} does not match the committed KAPPA={KAPPA}"
    f" (expected {_expected!r}). The backtest verdict above refers to a config"
    " this notebook does not use -- fix one or the other."
)


## Propose the next point

Unrestricted over all five axes, since none is flat.

The checks separate the two kinds of bound contact, and report convex-hull membership for information only -- at D=5 with 21 points nearly every observation is a hull vertex, so almost any proposal reads as outside it.

In [ ]:
with fitting("the committed proposal"):
    x_next, gp = generate_next_point(
        X, y, bounds,
        acquisition="ucb",
        kappa=KAPPA,        # see the comparison above
        maximize=True,
        n_restarts=60,
        random_state=0,
    )


mu, sigma = gp.predict(normalize(x_next.reshape(1, -1), bounds), return_std=True)
print(f"--- Next point to evaluate (bounds shape {bounds.shape}) ---")
print("x_next:", np.round(x_next, 6))
print(gp.kernel_)
print(f"GP predicted mean: {mu[0]:.6f}, predicted std: {sigma[0]:.6f}")
print(f"current best observed y: {y.max():.6f}"
      f"  -> predicted improvement: {mu[0] - y.max():+.6f}")

# Bounds are the TRUE DOMAIN, so neither edge is automatically a defect. What
# matters is whether the axis is monotone toward that edge (expected) or not
# (worth questioning). See bound_report's docstring and the axis cell.
up, lo = bound_report(x_next)
for label, dims, edge in (("UPPER", up, 1), ("lower", lo, 0)):
    print(f"\non the {label} domain edge:", dims or "none")
    for d in dims:
        sr = spearmanr(X[:, d], y)[0]
        toward = (sr < 0 and edge == 0) or (sr > 0 and edge == 1)
        if abs(sr) > 0.4 and toward:
            print(f"  x{d}: EXPECTED -- monotone (spearman {sr:+.3f}) with its best"
                  f" end at this edge")
        else:
            print(f"  x{d}: QUESTION IT -- spearman {sr:+.3f} does not point this way,"
                  " so the edge may be the optimiser stopping rather than an optimum")

outside = [d for d in range(D) if not (X[:, d].min() <= x_next[d] <= X[:, d].max())]
print("outside the observed range on its own axis:", outside or "none")
if hull is not None:
    print(f"inside the convex hull: {bool(hull.find_simplex(x_next) >= 0)}"
          "  (weak test at D=5 -- see above)")

print(f"\ndistance from the incumbent: {np.linalg.norm(x_next - incumbent):.6f}")
print("nearest 3 observations to x_next:")
for i in np.argsort(np.linalg.norm(X - x_next, axis=1))[:3]:
    print(f"  dist={np.linalg.norm(X[i] - x_next):.4f}  y={y[i]:+.6f}  X={np.round(X[i], 3)}")


## Visualise the GP and acquisition function via 1D slices

Each panel holds the other four dimensions fixed at the current best observed point and sweeps one dimension. Dotted line is the fixed centre, dashed red is the proposed `x_next`, green is UCB.

Unlike functions 3, 5, 7 and 8, expect **no flat panels** here -- every axis moves the mean by a comparable amount, which is why the search is unrestricted. The x0 panel should look the most structured (shortest length-scale) despite x0 having almost no monotone correlation with `y`.

This is a *partial* view: it shows the GP along each axis near the best point, not interactions between dimensions.

In [ ]:
plot_nd_slices(
    X, y, bounds, gp,
    acquisition_fn=ucb_acquisition,
    x_next=x_next,
    acq_kwargs={"kappa": KAPPA, "maximize": True},
)
plt.show()

## Sanity-check the surrogate model: leave-one-out calibration

Refits the GP once per observation, leaving it out, and predicts it from the rest. At D=5 this is the main way to judge the surrogate, since the fitted surface can't be inspected directly.

Read the error bars rather than the correlation. Unlike functions 4 and 5, the worst-predicted point here should be an *initial* observation rather than the newly collected one -- the new point was predicted to 0.6 sigma before it arrived, so it is not the model's weak spot.

In [ ]:
with fitting("leave-one-out calibration"):
    pred_mean, pred_std = loo_predictions(X, y, bounds, gp_kwargs={"n_restarts_optimizer": 20})

plot_loo_calibration(y, pred_mean, pred_std)
plt.show()

within = np.abs(y - pred_mean) <= 1.96 * pred_std
print(f"points inside their own 95% LOO interval: {within.sum()}/{len(y)}")
worst = int(np.argmax(np.abs(y - pred_mean)))
print(f"worst-predicted: index {worst} -> true {y[worst]:.6f},"
      f" predicted {pred_mean[worst]:.6f} (std {pred_std[worst]:.6f})")
src = "initial batch" if worst < n_initial else f"collected #{worst - n_initial + 1}"
print(f"  that point is from the {src}")


## Iteration diagnostics

`compute_iteration_diagnostics` replays the ordered `X`/`y` to reconstruct what the acquisition value, GP hyperparameters, and domain-wide uncertainty were at each past proposal -- no persisted log involved.

**`domain_grid_n` MUST be lowered past D=4.** Its `domain_mean_std` field averages the posterior std over a dense grid built as `domain_grid_n ** D` points, and the default `domain_grid_n=40` means:

| D | grid points at the default | array size |
|---|---|---|
| 4 (functions 4, 5) | 2.6e6 | 0.08 GB -- fine |
| **5 (this one)** | **1.0e8** | **4.1 GB -- kills the kernel** |
| 6 (function 7) | 4.1e9 | 197 GB |
| 8 (function 8) | 6.6e12 | 419,000 GB |

The first run of this notebook died exactly there. The cell below sizes the grid so `domain_grid_n ** D` stays around 200,000 regardless of `D`, which gives `domain_grid_n=11` here. That makes `domain_mean_std` a coarser estimate of domain-average uncertainty -- it is still a fair basis for comparing one iteration against another, but don't compare its absolute value against a run that used a different grid.

Two further caveats, as on the other notebooks. The setting is taken from `KAPPA` so it can't drift from what the proposal used -- but the single recorded observation came from the old week-1 sweep (`pi`/`xi=0.01`), so its replayed acquisition value describes a decision never made that way; treat that column as meaningless until the history is UCB throughout. And it assumes row order is true chronological order.

`plot_bo_diagnostics` is skipped (it hard-codes a 2D scatter panel and this is 5D); the trend plots below are dimension-agnostic and gated on having a few completed iterations.

In [ ]:
# compute_iteration_diagnostics builds a domain_grid_n ** D grid. The default of
# 40 is 40**5 = 1.0e8 points (~4 GB) at D=5 and WILL kill the kernel. Size it so
# the grid stays ~200k points whatever D is.
DOMAIN_GRID_N = max(3, int(200_000 ** (1.0 / D)))
print(f"domain_grid_n = {DOMAIN_GRID_N} -> {DOMAIN_GRID_N ** D:,} grid points"
      f" (the default 40 would be {40 ** D:,})")

# The old `if len(new_y) == 0` guard is gone: new_y is populated above, so that
# branch was unreachable. The count comes from the MODELLING set rather than
# len(new_y), so it stays correct if rows are ever excluded from the fit.
n_replayed = len(y) - n_initial

with fitting("iteration-diagnostics replay"):
    history = compute_iteration_diagnostics(X, y, bounds, n_initial=n_initial,
                                            acquisition="ucb", kappa=KAPPA,
                                            maximize=True,
                                            domain_grid_n=DOMAIN_GRID_N)

print("\nBest y so far:", np.nanmax(history["y"]))
print("completed iterations in the modelling set:", n_replayed)
print("\nCAVEAT: the history is MIXED. Week 2 was proposed under the previous")
print("template's settings, not ucb/kappa=%s, and compute_iteration_diagnostics"
      % KAPPA)
print("applies one setting to the whole history -- so its acq_value is an")
print("artefact of the mismatch, not a property of that point. The GP-hyperparameter")
print("and domain_mean_std columns do not depend on the acquisition and are fine.")

if n_replayed >= 3:
    for plot_fn in (plot_convergence, plot_acquisition_decay,
                    plot_uncertainty_shrinkage, plot_step_distance):
        plot_fn(history)
        plt.show()
else:
    print(f"\nOnly {n_replayed} replayed iteration(s) -- need at least 3 before the")
    print("trend plots say anything. Skipping them; the raw fields are below.")

## Raw diagnostic fields

In [ ]:
print("y:", np.round(history["y"], 5))
print("\niteration:", history["iteration"])
print("\nacq_value (NaN = initial batch; see the caveat above):", history["acq_value"])
print("\npred_mean at proposal time:", history["pred_mean"])
print("\ndomain_mean_std:", history["domain_mean_std"])

In [28]:
# The proposal as a hyphen-separated string, for submission.
print("-".join(f"{v:.6f}" for v in x_next))

# Full precision as well. 6 dp is fine to submit, but paste THIS into next
# week's new_X: a 6-dp copy of function 4's proposal rounded 4e-7 outside its
# own upper bound and tripped validate_bounds_consistency.
print("\nfull precision (use for next week's new_X):")
print("-".join(repr(float(v)) for v in x_next))

0.497078-0.264731-0.582415-0.764018-0.000000

full precision (use for next week's new_X):
0.49707819359032634-0.2647311219138715-0.5824149522356864-0.7640181811683764-0.0
